In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/processed/engine.db")

query = """
SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT unit) AS n_engines
FROM readings
GROUP BY split
"""
pd.read_sql(query, con)

,split,n_rows,n_engines
0,test,13096,100
1,train,20631,100


In [2]:
query = """
SELECT
    unit,
    cycle,
    MAX(cycle) OVER (PARTITION BY unit) - cycle AS rul
FROM readings
WHERE split = 'train'
ORDER BY unit, cycle
"""
labels = pd.read_sql(query, con)
labels.head(), labels.tail()

(   unit  cycle  rul
 0     1      1  191
 1     1      2  190
 2     1      3  189
 3     1      4  188
 4     1      5  187,
        unit  cycle  rul
 20626   100    196    4
 20627   100    197    3
 20628   100    198    2
 20629   100    199    1
 20630   100    200    0)

In [3]:
query = """
SELECT
    unit,
    cycle,
    s_11,
    AVG(s_11) OVER (
        PARTITION BY unit
        ORDER BY cycle
        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
    ) AS s_11_avg5
FROM readings
WHERE split = 'train'
ORDER BY unit, cycle
"""
roll = pd.read_sql(query, con)
roll.head(8)

,unit,cycle,s_11,s_11_avg5
0,1,1,47.47,47.470
1,1,2,47.49,47.480
2,1,3,47.27,47.410
3,1,4,47.13,47.340
4,1,5,47.28,47.328
5,1,6,47.16,47.266
6,1,7,47.36,47.240
7,1,8,47.24,47.234


In [4]:
sensors = ["s_2", "s_3", "s_4", "s_7", "s_8", "s_9", "s_11",
           "s_12", "s_13", "s_14", "s_15", "s_17", "s_20", "s_21"]

raw_cols = ", ".join(sensors)
avg_cols = ", ".join(
    f"AVG({s}) OVER (PARTITION BY split, unit ORDER BY cycle "
    f"ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS {s}_avg5"
    for s in sensors
)

con.execute("DROP TABLE IF EXISTS features")
con.execute(f"""
CREATE TABLE features AS
SELECT
    split, unit, cycle,
    CASE WHEN split = 'train'
         THEN MAX(cycle) OVER (PARTITION BY split, unit) - cycle
    END AS rul,
    {raw_cols},
    {avg_cols}
FROM readings
""")
con.commit()

check = pd.read_sql("SELECT split, COUNT(*) AS n_rows, MAX(rul) AS max_rul FROM features GROUP BY split", con)
check

,split,n_rows,max_rul
0,test,13096,NaN
1,train,20631,361.0
